<a href="https://colab.research.google.com/github/kyuuf/Youtube_Comment_Crawler/blob/master/%EC%9C%A0%ED%88%AC%EB%B8%8C_%EB%8C%93%EA%B8%80_%ED%81%AC%EB%A1%A4%EB%A7%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필수설치 파일

In [ ]:
!pip install google-api-python-client pandas

from googleapiclient.discovery import build
import pandas as pd
import time

YouTube Comment Scraper for Google Colab

In [ ]:

!pip install google-api-python-client pandas

from googleapiclient.discovery import build
import pandas as pd
import time
from google.colab import files

In [ ]:
import re
import time
import pandas as pd
from googleapiclient.discovery import build
from google.colab import files

# === ✅ 설정 ===
API_KEY = ""     # 구글 클라우드 콘솔에서 발급받은 YouTube API 키
VIDEO_ID = "stwwJ5vRLLY"   # 유튜브 영상 ID (예: https://youtu.be/dQw4w9WgXcQ → dQw4w9WgXcQ)
OUTPUT_FILE = "youtube_all_comments.csv"

# 🔧 추가: 댓글을 한 줄로 정리하는 함수
def normalize_comment(text):
    if pd.isna(text):
        return text
    text = str(text)
    # 줄바꿈, 탭, 여러 칸 공백을 모두 '공백 1개'로 통일
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# === YouTube API 연결 ===
youtube = build('youtube', 'v3', developerKey=API_KEY)

comments_data = []
next_page_token = None

print("유튜브 댓글을 가져오는 중입니다...")

while True:
    request = youtube.commentThreads().list(
        part="snippet,replies",
        videoId=VIDEO_ID,
        maxResults=100,
        pageToken=next_page_token,
        textFormat="plainText"
    )
    response = request.execute()

    for item in response['items']:
        top_comment = item['snippet']['topLevelComment']['snippet']
        comments_data.append({
            "comment_id": item['id'],
            "parent_id": None,
            "author": top_comment['authorDisplayName'],
            "text": top_comment['textDisplay'],
            "likes": top_comment['likeCount'],
            "published_at": top_comment['publishedAt']
        })

        # ✅ 대댓글(reply)도 포함
        if 'replies' in item:
            for reply in item['replies']['comments']:
                reply_snippet = reply['snippet']
                comments_data.append({
                    "comment_id": reply['id'],
                    "parent_id": reply_snippet['parentId'],
                    "author": reply_snippet['authorDisplayName'],
                    "text": reply_snippet['textDisplay'],
                    "likes": reply_snippet['likeCount'],
                    "published_at": reply_snippet['publishedAt']
                })

    next_page_token = response.get('nextPageToken')
    if not next_page_token:
        break

    time.sleep(1)

# === DataFrame으로 변환 ===
df = pd.DataFrame(comments_data)

# 🔧 추가: 줄바꿈/여러 칸 공백을 정리해서 한 줄로 만들기
df['text'] = df['text'].apply(normalize_comment)

# === CSV로 저장 ===
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')  # 🔥 한글 깨짐 방지 (UTF-8-SIG)

print(f"✅ {len(df)}개의 댓글을 수집하여 '{OUTPUT_FILE}' 파일로 저장했습니다.")

# === Colab에서 CSV 파일 다운로드 ===
files.download(OUTPUT_FILE)
print("다운로드가 자동으로 시작됩니다.")


유튜브 댓글을 가져오는 중입니다...
✅ 2925개의 댓글을 수집하여 'youtube_all_comments.csv' 파일로 저장했습니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드가 자동으로 시작됩니다.


불필요한 아이디 및 링크 제외

In [ ]:
import pandas as pd
import re

# 1. 파일 불러오기 (파일명을 본인의 파일명으로 수정하세요)


def clean_text(text):
    if not isinstance(text, str):
        return ""
    # @아이디 형태 제거
    text = re.sub(r'@\w+', '', text)
    # URL 제거
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # 특수문자 제거 및 양끝 공백 제거
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

# 'content'는 댓글이 들어있는 열 이름입니다.
df['cleaned_content'] = df['text'].apply(clean_text) # Column name corrected to 'text'
df.head()

# CSV 저장
df.to_csv("youtube_all_commentssda.csv", index=False, encoding="utf-8-sig")

# 다운로드
from google.colab import files
files.download("youtube_all_commentssd.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import re

# 1. 파일 불러오기 (파일명을 본인의 파일명으로 수정하세요)


def clean_text(text):
    if not isinstance(text, str):
        return ""
    # @아이디 형태 제거
    text = re.sub(r'@\w+', '', text)
    # URL 제거
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # 특수문자 제거 및 양끝 공백 제거
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

# 'content'는 댓글이 들어있는 열 이름입니다.
df['cleaned_content'] = df['text'].apply(clean_text) # Column name corrected to 'text'
df.head()

# CSV 저장
df.to_csv("son_eng_comments_cleaned.csv", index=False, encoding="utf-8-sig")

# 다운로드
from google.colab import files
files.download("son_eng_comments_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>